In [1]:
import argparse
import os
import cv2 
import skvideo.io

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns 

from easydict import EasyDict

import torch
from torch.utils.data import DataLoader

from posetail.datasets.datasets import Rat7mDataset, Rat7mIterableDataset, custom_collate_2d
from train_utils import *

%load_ext autoreload
%autoreload 2

In [2]:

def load_predictions(data_path, device):

    data = np.load(data_path) 

    coords_pred = torch.from_numpy(data['coords_pred']).to(device)
    vis_pred = torch.from_numpy(data['vis_pred']).to(device)
    conf_pred = torch.from_numpy(data['conf_pred']).to(device)

    coords_true = torch.from_numpy(data['coords_true']).to(device)
    vis_true = torch.from_numpy(data['vis_true']).to(device)

    fnums = torch.from_numpy(data['fnums']).to(device)
    video_path = ''.join(data['video_path'])

    return coords_pred, vis_pred, conf_pred, coords_true, vis_true, fnums, video_path


def get_video_predictions(video_path, model, dataloader, outdir, debug_ix = -1):

    device = model.device
    model.eval()

    start_time = time.time()
    timestamp = get_timestamp()

    coords_pred = []
    vis_pred = []
    conf_pred = []
    coords_true = []
    vis_true = []
    fnums = []

    video_name = os.path.splitext(os.path.basename(video_path))[0]

    for j, batch in enumerate(dataloader):

        views = [view.to(device) for view in batch.views]
        coords = batch.coords.to(device)
        fnum = batch.fnums.to(device)

        if j == debug_ix: 
            break

        cgroup = None 
        if 'cgroup' in batch: 
            cgroup = batch.cgroup

        vis = get_vis_true(coords)
                      
        # get model predictions
        with torch.no_grad():
            coords_p, vis_p, conf_p, *_ = model(
                views = views, 
                coords = coords[:, 0, ...], 
                camera_group = cgroup, 
                offset_dict = None
            )

        coords_pred.append(torch.squeeze(coords_p, dim = 0))
        vis_pred.append(torch.squeeze(vis_p, dim = 0))
        conf_pred.append(torch.squeeze(conf_p, dim = 0))
        coords_true.append(torch.squeeze(coords, dim = 0))
        vis_true.append(torch.squeeze(vis, dim = 0))
        fnums.append(torch.squeeze(fnum, dim = 0))

    coords_pred = torch.cat(coords_pred, dim = 0)
    vis_pred = torch.cat(vis_pred, dim = 0)
    conf_pred = torch.cat(conf_pred, dim = 0)
    coords_true = torch.cat(coords_true, dim = 0)
    vis_true = torch.cat(vis_true, dim = 0)
    fnums = torch.cat(fnums, dim = 0)

    results_path = os.path.join(outdir, f'{video_name}_predictions.npz')
    elapsed_time = time.time() - start_time
    elapsed_time_hms = str(timedelta(seconds = elapsed_time)).split('.')[0]

    np.savez(results_path,
        coords_pred = coords_pred.cpu(), 
        vis_pred = vis_pred.cpu(), 
        conf_pred = conf_pred.cpu(),
        coords_true = coords_true.cpu(),
        vis_true = vis_true.cpu(),
        fnums = fnums.cpu(), 
        video_path = list(video_path), 
        elapsed_time = list(np.array([elapsed_time])), 
        elapsed_time_hms = list(elapsed_time_hms))

    return results_path

def main(config_path, model_path, video_path, data_path, outdir): 

    config = load_config(config_path)
    model = load_checkpoint(config_path, model_path)
    model.eval()

    set_seeds(config.training.seed)

    dataset = Rat7mDataset(
        video_path = video_path, 
        data_path = data_path, 
        n_frames = config.dataset.test.n_frames)

    #dataset = get_dataset(**config.dataset.train)

    dataloader = DataLoader(
        dataset, 
        batch_size = config.dataset.batch_size, 
        collate_fn = custom_collate_2d)

    outpath = safe_make('results')

    results_path = get_video_predictions(video_path, 
        model, dataloader, outdir, debug_ix = -1)
        
    print(f'predictions saved to {results_path}')




In [3]:
# testing iterable dataset
# run_id = 'run-20250303_121158-17z0mb1w'
run_id = 'run-20250423_201321-kmadlx6h'
config_path = f'/allen/aind/scratch/katie.rupp/wandb/{run_id}/files/config.toml'
config = load_config(config_path)
device = (torch.device('cuda') if torch.cuda.is_available() else 'cpu')

video_path = '/allen/aind/scratch/katie.rupp/data/rat7m/videos/s5-d2/s5-d2-camera1-0.mp4'
data_path = '/allen/aind/scratch/katie.rupp/data/rat7m/data/mocap-s5-d2.mat'

# iterable dataset
dataset = get_dataset(**config.dataset.train)

# non-iterable dataset
# dataset = Rat7mDataset(
#     video_path = video_path, 
#     data_path = data_path, 
#     n_frames = config.dataset.test.n_frames)

dataloader = DataLoader(
    dataset, 
    batch_size = config.dataset.batch_size, 
    collate_fn = custom_collate_2d)

for j, batch in enumerate(dataloader):

    views = [view.to(device) for view in batch.views]
    coords = batch.coords.to(device)
    fnum = batch.fnums.to(device)
    break

In [ ]:
fig_path = safe_make('figures')

coord = coords[0, 0, :, :].detach().cpu().numpy().astype(int)
frame = views[0][0, 0, :, :, :].detach().cpu().numpy()

for xy in coord:
    cv2.circle(frame, tuple(xy), 5, (0, 255, 0), -1)

outpath = os.path.join(fig_path, f'out0.png')
cv2.imwrite(outpath, frame)

print(frame.shape)

frame = frame 
plt.imshow(frame)
plt.show()

In [ ]:
# load model and set to eval mode for inference 
# run_id = 'run-20250423_204116-cy2uip29'
# run_id = 'run-20250423_201321-kmadlx6h'
# run_id = 'run-20250430_043024-dj28ehws'
run_id = 'run-20250429_130204-uz145qcz'
config_path = f'/allen/aind/scratch/katie.rupp/wandb/{run_id}/files/config.toml'
model_path = f'/allen/aind/scratch/katie.rupp/wandb/{run_id}/files/checkpoints/checkpoint_000299.pth'

cam = 1
video_path = f'/allen/aind/scratch/katie.rupp/data/rat7m/videos/s5-d2/s5-d2-camera{cam}-0.mp4'
data_path = '/allen/aind/scratch/katie.rupp/data/rat7m/data/mocap-s5-d2.mat'
outdir = '/home/katie.rupp/posetail/results'
main(config_path, model_path, video_path, data_path, outdir)

In [9]:
video_path = f'/allen/aind/scratch/katie.rupp/data/rat7m/videos/s5-d2/s5-d2-camera{cam}-0.mp4'
results_path = f'/home/katie.rupp/posetail/results/s5-d2-camera{cam}-0_predictions.npz'
device = (torch.device('cuda') if torch.cuda.is_available() else 'cpu')
coords_pred, vis_pred, conf_pred, coords_true, vis_true, fnums, video_path = load_predictions(results_path, device)
fig_path = safe_make('figures')

coords_true = coords_true.cpu().numpy().astype(int) 
coords_pred = coords_pred.cpu().numpy().astype(int) * 2

i = 0
j = 0
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = 30.0 # cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
runid = run_id.split('-')[-1]
out = cv2.VideoWriter(os.path.join(fig_path, f'output{cam}_{runid}.mp4'), fourcc, fps, (frame_width, frame_height))
ret = True

while ret:

    ret, frame = cap.read() 

    if not ret: 
        break
    
    if i not in fnums:
        out.write(frame)

    else:
        for coord_true, coord_pred in zip(coords_true[j], coords_pred[j]):
            cv2.circle(frame, tuple(coord_true), 5, (0, 255, 0), -1)
            cv2.circle(frame, tuple(coord_pred), 5, (255, 0, 0), -1)

        # outpath = os.path.join(fig_path, f'out{i}.png')
        # cv2.imwrite(os.path.join(fig_path, f'out{i}.png'), frame)
        out.write(frame)
        j += 1

    i += 1

cap.release()
out.release()


In [ ]:
video_path = '/allen/aind/scratch/katie.rupp/data/rat7m/videos/s5-d2/s5-d2-camera1-0.mp4'
results_path = '/home/katie.rupp/posetail/results/s5-d2-camera1-0_predictions.npz'
device = (torch.device('cuda') if torch.cuda.is_available() else 'cpu')
coords_pred, vis_pred, conf_pred, coords_true, vis_true, fnums, video_path = load_predictions(results_path, device)
fig_path = safe_make('figures')

coords_true = coords_true.cpu().numpy().astype(np.uint8)
coords_pred = coords_pred.cpu().numpy().astype(np.uint8)

i = 0
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = round(cap.get(cv2.CAP_PROP_FPS), 2)

outname = os.path.join(fig_path, 'output2.avi')
ret = True

writer = skvideo.io.FFmpegWriter(outname, inputdict={
        # '-hwaccel': 'auto',
        '-framerate': str(fps),
    }, outputdict={
        '-vcodec': 'h264', '-qp': '28',
        '-pix_fmt': 'yuv420p', # to support more players
        '-vf': 'pad=ceil(iw/2)*2:ceil(ih/2)*2' # to handle width/height not divisible by 2
})

while ret:

    ret, frame = cap.read() 
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    if i not in fnums:
        writer.writeFrame(frame)
        continue
    
    if not ret: 
        break

    for coord in coords_true[i]:
        # print(coord)
        cv2.circle(frame, tuple(coord), 5, (0, 255, 0), -1)

    for coord in coords_pred[i]: 
        cv2.circle(frame, tuple(coord), 5, (255, 0, 0), -1)

    # outpath = os.path.join(fig_path, f'out{i}.png')
    # cv2.imwrite(os.path.join(fig_path, f'out{i}.png'), frame)
    writer.writeFrame(frame)
    i += 1
    print(i)

cap.release()
writer.close()
